In [ ]:
# Name: Abdul Muhaimin bin Abd Manaf
# Student ID: SW01082844
# Lab Assignment 1 - Walmart Reviews Scraper using ScraperAPI

import requests
import pandas as pd
import time
from datetime import datetime
import json
import os

# Used ScraperAPI to avoid existing proxies and timeout errors
SCRAPERAPI_KEY = "be3dad8fe53f2e8303be194326ce27e5"

def fetch_with_retry(url, params, max_retries=3, timeout=60):
    # This function handles requests with automatic retry when they fail
    # It will try up to max_retries times if there's a timeout or error
    
    for attempt in range(1, max_retries + 1):
        try:
            print(f"    Attempt {attempt}/{max_retries}...")
            response = requests.get(url, params=params, timeout=timeout)
            response.raise_for_status()
            return response
        except requests.exceptions.Timeout:
            print(f"    Timeout on attempt {attempt}")
            if attempt < max_retries:
                wait_time = 2 ** attempt  # Waits 2, then 4, then 8 seconds
                print(f"    Waiting {wait_time} seconds before retry...")
                time.sleep(wait_time)
            else:
                print(f"    All {max_retries} attempts failed due to timeout")
                return None
        except requests.exceptions.RequestException as e:
            print(f"    Error on attempt {attempt}: {e}")
            if attempt < max_retries:
                wait_time = 2
                print(f"    Waiting {wait_time} seconds before retry...")
                time.sleep(wait_time)
            else:
                return None
    
    return None

def fetch_walmart_reviews_scraperapi(product_id, max_pages=5):
    # Main function to scrape Walmart reviews using ScraperAPI
    # Arguments:
    #   product_id: The Walmart product ID (like "6288916776")
    #   max_pages: Maximum number of pages to scrape (default 5)
    # Returns:
    #   A pandas DataFrame containing all the scraped reviews
    
    all_reviews = []
    failed_pages = []
    
    print(f"\n{'='*60}")
    print(f"Starting Walmart Reviews Scraper using ScraperAPI")
    print(f"Product ID: {product_id}")
    print(f"Max Pages: {max_pages}")
    print(f"Timeout: 60 seconds with 3 retry attempts")
    print(f"{'='*60}\n")
    
    # Loop through each page from 1 to max_pages
    for page in range(1, max_pages + 1):
        print(f"\n--- Scraping Page {page} ---")
        
        # Prepare the parameters for ScraperAPI
        # These tell ScraperAPI what data we want
        payload = {
            'api_key': SCRAPERAPI_KEY,      # Your API key for authentication
            'product_id': product_id,        # The Walmart product ID
            'page': page,                     # Which page of reviews to get
            'country_code': 'us',             # Get reviews from US customers
            'render': True,                    # Enable JavaScript rendering
            'premium': True                     # Use premium proxies for better success
        }
        
        # Use the retry function to fetch data from ScraperAPI
        response = fetch_with_retry(
            'https://api.scraperapi.com/structured/walmart/review',
            payload,
            max_retries=3,
            timeout=60
        )
        
        # If we couldn't get a response after retries, mark this page as failed
        if not response:
            print(f"  Failed to fetch page {page} after multiple attempts")
            failed_pages.append(page)
            
            # Continue to next page instead of stopping completely
            if page < max_pages:
                print(f"  Continuing to next page...")
                time.sleep(2)
            continue
        
        try:
            # Parse the JSON response from ScraperAPI
            data = response.json()
            
            # Check if we got valid data
            if not data:
                print(f"  No data returned for page {page}")
                failed_pages.append(page)
                continue
            
            # Extract the reviews from the response
            if 'reviews' in data and data['reviews']:
                reviews_list = data['reviews']
                print(f"  Found {len(reviews_list)} reviews on page {page}")
                
                # Process each individual review
                page_review_count = 0
                for review in reviews_list:
                    # Extract reviewer name (if not found, use 'N/A')
                    reviewer_name = review.get('author', 'N/A')
                    
                    # Extract review date (if not found, use 'N/A')
                    review_date = review.get('date_published', 'N/A')
                    
                    # Extract review title and text
                    review_title = review.get('title', '')
                    review_text = review.get('text', '')
                    
                    # Combine title and text to create full review content
                    if review_title and review_text:
                        review_content = f"{review_title}: {review_text}"
                    elif review_title:
                        review_content = review_title
                    elif review_text:
                        review_content = review_text
                    else:
                        review_content = 'N/A'
                    
                    # Create a dictionary with ONLY the three required fields
                    # This matches the assignment requirements
                    review_data = {
                        'Reviewer_Name': reviewer_name,
                        'Review_Date': review_date,
                        'Review_Content': review_content
                    }
                    
                    # Add this review to our collection
                    all_reviews.append(review_data)
                    page_review_count += 1
                    
                    # Show a preview of the first few reviews
                    if len(all_reviews) <= 5:
                        preview = review_content[:60] + "..." if len(review_content) > 60 else review_content
                        print(f"    {reviewer_name[:15]} - {review_date} - {preview}")
                
                print(f"  Page {page} complete. Found {page_review_count} reviews.")
            else:
                print(f"  No reviews found on page {page}")
                failed_pages.append(page)
            
        except json.JSONDecodeError as e:
            # Handle case where response isn't valid JSON
            print(f"  Error parsing JSON on page {page}: {e}")
            failed_pages.append(page)
        except Exception as e:
            # Handle any other unexpected errors
            print(f"  Unexpected error on page {page}: {e}")
            failed_pages.append(page)
        
        # Wait between pages to be respectful to the API
        if page < max_pages:
            # Use longer delay after failed pages
            delay = 3 if page in failed_pages else 2
            print(f"  Waiting {delay} seconds before next request...")
            time.sleep(delay)
    
    # Print summary of any pages that failed
    if failed_pages:
        print(f"\nFailed to fetch pages: {failed_pages}")
    
    # Convert the list of reviews to a pandas DataFrame
    return pd.DataFrame(all_reviews)

def save_to_csv(dataframe, filename="walmart_reviews.csv"):
    # Save DataFrame to CSV with fixed filename
    # Arguments:
    #   dataframe: The pandas DataFrame to save
    #   filename: The name of the file to save 
    # Returns:
    #   The filename that was used
    
    # Save to CSV without the index column
    dataframe.to_csv(filename, index=False, encoding='utf-8')
    print(f"\nData saved to {filename}")
    return filename

def test_api_connection():
    # Test if ScraperAPI is working before starting the main scrape to save time if API key is invalid
    
    print("\n--- Testing ScraperAPI Connection ---")
    
    # Simple test to check IP address through ScraperAPI
    test_payload = {
        'api_key': SCRAPERAPI_KEY,
        'url': 'https://httpbin.org/ip',
        'premium': True
    }
    
    response = fetch_with_retry('https://api.scraperapi.com/', test_payload, max_retries=2, timeout=20)
    
    if response:
        print("ScraperAPI connection successful!")
        print(f"   Response: {response.text[:100]}")
        return True
    else:
        print("ScraperAPI connection failed after retries")
        return False

def main():
    # Main function to execute the entire scraping process
    
    print("WALMART REVIEWS SCRAPER USING SCRAPERAPI")
    print("=" * 60)
    print("Name: Abdul Muhaimin bin Abd Manaf")
    print("Student ID: SW01082844")
    print("=" * 60)
    
    # To check if the API key placeholder has been replaced
    if SCRAPERAPI_KEY == "YOUR_SCRAPERAPI_API_KEY":
        print("\nPlease set your ScraperAPI key in the script!")
        print("Get your key from: https://www.scraperapi.com")
        return
    
    # Test the API connection first
    if not test_api_connection():
        print("\nAPI connection test failed. Please check your API key.")
        return
    
    # Product ID from the Walmart URL
    PRODUCT_ID = "6288916776"
    MAX_PAGES = 5
    
    print(f"\nProduct ID: {PRODUCT_ID}")
    print(f"Pages to scrape: {MAX_PAGES}")
    print("-" * 60)
    
    # Scrape the reviews
    reviews_df = fetch_walmart_reviews_scraperapi(PRODUCT_ID, MAX_PAGES)
    
    # Display results if we got any reviews
    if not reviews_df.empty:
        print("\n" + "=" * 60)
        print("SAMPLE DATA (First 10 reviews):")
        print("=" * 60)
        
        # Display only the three required columns
        display_cols = ['Reviewer_Name', 'Review_Date', 'Review_Content']
        display_df = reviews_df[display_cols].head(10)
        print(display_df.to_string(index=False))
        
        # Save the reviews to a CSV file
        filename = save_to_csv(reviews_df)
        
        # Show where the file was saved
        file_path = os.path.abspath(filename)
        print(f"\nFile saved at: {file_path}")
        
        # Print simple statistics
        print(f"\nSummary:")
        print(f"  - Total reviews: {len(reviews_df)}")
        
    else:
        print("\nNo reviews were scraped.")

# This ensures main() runs only when the script is executed directly
if __name__ == "__main__":
    main()

WALMART REVIEWS SCRAPER USING SCRAPERAPI
Name: Abdul Muhaimin bin Abd Manaf
Student ID: SW01082844

--- Testing ScraperAPI Connection ---
    Attempt 1/2...
ScraperAPI connection successful!
   Response: {
  "origin": "145.82.145.177"
}


Product ID: 6288916776
Pages to scrape: 5
------------------------------------------------------------

Starting Walmart Reviews Scraper using ScraperAPI
Product ID: 6288916776
Max Pages: 5
Timeout: 60 seconds with 3 retry attempts


--- Scraping Page 1 ---
    Attempt 1/3...
  Found 10 reviews on page 1
    Sam - 7/29/2025 - Great lifting straps: Found these on sale, they typically go...
    debra - 1/28/2026 - They wear a christmas present.My grandson loves them
    Justin Schauer - 4/4/2025 - Lifting straps: I very very much like the lifting straps by ...
    Tanner Opheiken - 12/10/2024 - Ah yes, my life is good: Haven't ever used straps before. A ...
    Preston Fraley - 3/10/2025 - Lifting straps: Good feeling with these straps. Was able to ...
